## Регрессия для SI

In [1]:
!pip install catboost -q
!pip install xgboost -q
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.svm import SVR
import xgboost as xgb

from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42

In [2]:
data = pd.read_csv('data/preprocessed_data.csv')

In [3]:
# Подготовим данные
ex_colums = ['log_IC50', 'log_CC50', 'log_SI', 'IC50, mM', 'CC50, mM', 'SI',
                'IC50_median', 'CC50_median', 'SI_median', 'SI_8']

X = data.drop(columns=ex_colums)
y = data['log_SI']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,          
    random_state=RANDOM_STATE
)
X_train.shape

(798, 30)

In [4]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

pipline = Pipeline([('scaler', 'passthrough'), ('models', LinearRegression())])

In [5]:
# Параметры для различных моделей
param_grid = [
    {
        'scaler': [StandardScaler(), MinMaxScaler(), 'passthrough'],
        'models': [LinearRegression()]
    },
    {
        'scaler': [StandardScaler(), MinMaxScaler()],
        'models': [SVR()],
        'models__C': [0.1, 1, 10],
        'models__kernel': ['linear', 'rbf', 'sigmoid']
    },
    {
        'scaler': ['passthrough'],
        'models': [RandomForestRegressor(random_state=RANDOM_STATE)],
        'models__n_estimators': [100, 150, 300],
        'models__max_depth': [None, 30, 40],
        'models__min_samples_split': [2, 5, 10],
        'models__min_samples_leaf': [1, 3, 5]
    },
    {
        'scaler': ['passthrough'],
        'models': [xgb.XGBRegressor(random_state=RANDOM_STATE, verbosity=0)],
        'models__max_depth': [3, 6, 10],
        'models__n_estimators': [100, 200, 300],
        'models__learning_rate': [0.01, 0.1, 0.05],
    },
    {
       'scaler': ['passthrough'],
       'models': [CatBoostRegressor(random_state=RANDOM_STATE, verbose=0)],
       'models__iterations': [100, 200, 500, 1000],
       'models__learning_rate': [0.01, 0.1],
       'models__depth': [5, 10, 15]
    }
]

In [6]:
# Для подбора гиперпараметров воспользуемся RandomizedSearchCV
random_search = RandomizedSearchCV(
    pipline,
    param_distributions=param_grid,
    n_iter=30,
    cv=cv,
    scoring='neg_mean_squared_error',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

In [8]:
random_search.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


RandomizedSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('scaler', 'passthrough'),
                                             ('models', LinearRegression())]),
                   n_iter=30, n_jobs=-1,
                   param_distributions=[{'models': [LinearRegression()],
                                         'scaler': [StandardScaler(),
                                                    MinMaxScaler(),
                                                    'passthrough']},
                                        {'models': [SVR()],
                                         'models__C': [0.1, 1, 10],
                                         'models__kernel': ['linear',...
                                         'models__max_depth': [3, 6, 10],
                                         'models__n_estimators': [100, 200,
                                                                  300],
                                         'scaler': ['passthrough']},
                                        {'models': [CatBoostRegressor(loss_function='RMSE', random_state=42, verbose=0)],
                                         'models__depth': [5, 10, 15],
                                         'models__iterations': [100, 200, 500,
                                                                1000],
                                         'models__learning_rate': [0.01, 0.1],
                                         'scaler': ['passthrough']}],
                   random_state=42, scoring='neg_mean_squared_error',
                   verbose=1)

In [9]:
print('Лучшая модель и её параметры:\n\n', random_search.best_params_) 
print('Метрика  для лучшей модели:\n', random_search.best_score_) 

Лучшая модель и её параметры:

 {'scaler': 'passthrough', 'models__learning_rate': 0.01, 'models__iterations': 1000, 'models__depth': 5, 'models': CatBoostRegressor(loss_function='RMSE', random_state=42, verbose=0)}
Метрика  для лучшей модели:
 -0.40294576351647393


In [10]:
best_model = random_search.best_estimator_

# Оценим на тестовой выборке, посмотрим на несколько метрик
y_pred_log = best_model.predict(X_test)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))
mae_log = mean_absolute_error(y_test, y_pred_log)
r2_log = r2_score(y_test, y_pred_log)

print(f'rmse_log: {rmse_log}')
print(f'mae_log: {mae_log}')
print(f'r2_log: {r2_log}')

rmse_log: 0.7243781162526073
mae_log: 0.5343784844857797
r2_log: 0.23985606029173978


In [13]:
# Посмотрим на метрики без логарифма
y_pred = 10 ** y_pred_log
y_test_ic50 = data.loc[y_test.index, 'CC50, mM']
rmse = np.sqrt(mean_squared_error(y_test_ic50, y_pred))
mae = mean_absolute_error(y_test_ic50, y_pred)
r2 = r2_score(y_test_ic50, y_pred)

print(f'rmse: {rmse}')
print(f'mae: {mae}')
print(f'r2: {r2}')

rmse: 925.9401667943853
mae: 623.4101923862595
r2: -0.8200085676713089


In [11]:
data['SI'].describe()

count      998.000000
mean        72.650005
std        685.504279
min          0.011489
25%          1.457233
50%          3.856410
75%         16.525000
max      15620.600000
Name: SI, dtype: float64

In [12]:
# Посмторим насколько лучше полученная модель, чем просто заполнение средним
y_pred_log_mean = np.full_like(y_test, y_train.mean())
rmse_log_mean = np.sqrt(mean_squared_error(y_test, y_pred_log_mean))
print(f'RMSE по среднему: {rmse_log_mean}')

RMSE по среднему: 0.831298082350611


In [14]:
# Сравнение метрик моделей
res = pd.DataFrame(random_search.cv_results_)
res['model'] = res['params'].apply(lambda x: type(x['models']).__name__)
res['rmse'] = np.sqrt(-res['mean_test_score'])
res.groupby('model')['rmse'].agg(['mean', 'min', 'max'])

,mean,min,max
model,,,
CatBoostRegressor,0.649458,0.634780,0.658073
RandomForestRegressor,0.639736,0.637701,0.642532
SVR,11.177004,0.644407,53.202414
XGBRegressor,0.656338,0.646141,0.665219


In [15]:
# LinearRegression не отобразился в таблице, поэтому явно вычислим его RMSE
lin = LinearRegression()
lin.fit(X_train, y_train)
y_lin_pred = lin.predict(X_test)
np.sqrt(mean_squared_error(y_test, y_lin_pred))

0.7930416337608613

В ходе выполнения задачи регрессии для CC50 были рассмотрены  несколько моделей с разными гиперпараметрами (LinearRegression(), RandomForestRegressor(), CatBoostRegressor(), SVR(), XGBRegressor()) . Используя RandomizedSearchCV следующая модель была определена как лучшая: {'scaler': 'passthrough', 'models__learning_rate': 0.01, 'models__iterations': 1000, 'models__depth': 5, 'models': CatBoostRegressor(loss_function='RMSE', random_state=42, verbose=0)}.

RandomForestRegressor и CatBoostRegressor показали схожие результаты, в SVR опять виден большой разброс. 

Метрика в логарифмической шкале:

rmse_log: 0.7243781162526073
mae_log: 0.5343784844857797
r2_log: 0.23985606029173978

Модель объясняет 24% дисперсии логарифма SI, это означает, что сложность предсказания производного показателя велика.

Метрика в исходной шкале:

rmse: 925.9401667943853
mae: 623.4101923862595
r2: -0.8200085676713089

В исходной шкале модель показала отрицательный r2, то есть хуже простого прогноза средним. Это говорит о том, что точное количественное предсказание SI в натуральных единицах затруднено, поэтому для задач отбора соединений нужно использовать классификационные модели. Также, сравнив полученную модель с простым заполнением средним, видно, что ошибка у CatBoostRegressor  меньше - модель показывает лучшие результаты, чем простое предсказание средним.

В качестве рекомендации по улучшение можно сначала вычислить IC50, CC50, а потом уже получить SI, как их отношение.